# Carta Social: data collection

Inspect configured service records and their equipment URL index, and optionally
collect a bounded range of equipment IDs from the Carta Social portal.
Extraction and persistence are implemented in scripts/scraper.py.

The default mode makes no HTTP requests. Data paths come from root config.toml.
This notebook loads root .env before using the configured HTTP identity.

Each record describes a social service offered by a facility. A facility may
offer several services, so service and facility counts differ.


In [ ]:
from pathlib import Path
import sys

# Support execution from the repository root or the notebooks directory.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

if not (project_root / "scripts" / "utils.py").exists():
    raise FileNotFoundError("Start Jupyter from the ScraperCartaSocial repository.")

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


In [ ]:
import os
import pandas as pd

from scripts.utils import RAW, INPUTS, ensure_data_directories, load_environment
from scripts.scraper import run_scraper
from scripts.cleaning import standardize_columns

load_environment()
ensure_data_directories()


## Collection scope

Equipment identifiers provide access to portal records; they do not define
a random sample or establish complete coverage. Gaps, failed requests, and
portal changes must be distinguished from the absence of a facility.

Before enabling collection, review the portal's terms and select an
appropriate request rate. The initial range is limited to five IDs.
The HTML selectors originate from the supplied implementation and may
need revision if the portal changes.


In [ ]:
collect_new_data = False
first_id = 1
last_id = 5
request_delay_seconds = 5


## Optional collection

New HTML pages and JSON snapshots are stored in `raw/scraping/`;
per-ID state is stored separately in `interim/scraper/`. Existing
snapshots are reused, while failed requests are retried on subsequent runs.
The CSV is rebuilt from JSON snapshots to avoid duplicate appends.

Source snapshots remain separate from generated working tables. All data
paths resolve under the external data root configured in config.toml.


In [ ]:
if collect_new_data:
    new_records = run_scraper(
        start_id=first_id,
        end_id=last_id,
        sleep_interval=request_delay_seconds,
        user_agent=os.getenv("HTTP_USER_AGENT") or "CartaSocialPipeline/1.0",
    )
    print("Service records in the new collection:", len(new_records))
else:
    print("Using the existing local collection.")


## Source inspection

The configured service CSV is read as UTF-8, with equipment IDs treated as
text. Field labels are standardized to English for inspection; source values
and the input file remain unchanged.

To inspect a new scrape, point inputs.services in config.toml to the generated
raw/scraping/equipments.csv and restart the kernel.


In [ ]:
source_file = INPUTS["services"]
source_records = pd.read_csv(source_file, dtype={"Equipment ID": "string"})
services = standardize_columns(source_records)
services.head()


In [ ]:
service_count = len(services)
equipment_count = services["equipment_id"].nunique()

print("Service records:", service_count)
print("Distinct equipment:", equipment_count)


## Initial quality checks

Missing values and exact duplicate rows are reported without automatic
removal. Repeated equipment IDs are expected when a facility offers more
than one service.

Source update dates and collection timestamps represent different events.
The input's provenance should establish which dates are available before
they are used to interpret change over time.


In [ ]:
missing_values = services.isna().sum()
exact_duplicate_count = services.duplicated().sum()

print(missing_values)
print("Exact duplicate rows:", exact_duplicate_count)


## Equipment URL index

The configured URL CSV contains equipment_id and url. The ID is validated
against the URL query parameter. The main service CSV remains unchanged.
Several service records can refer to one equipment URL.

An optional text URL log can be cross-checked through inputs.url_text.
Coverage is computed below without contacting the source portal.

New scraper runs retain each URL in its JSON snapshot and a separate
equipment_urls.csv. Configure inputs.equipment_urls to read that export.


In [ ]:
from scripts.source_urls import load_equipment_urls

url_index = load_equipment_urls(
    csv_path=INPUTS["equipment_urls"],
    txt_path=INPUTS["url_text"],
)
service_equipment_ids = services["equipment_id"].drop_duplicates()
missing_url_ids = service_equipment_ids[~service_equipment_ids.isin(url_index["equipment_id"])]
unmatched_url_ids = url_index[~url_index["equipment_id"].isin(service_equipment_ids)]

print("Equipment URLs:", len(url_index))
print("Service equipment IDs without a URL:", len(missing_url_ids))
print("URL IDs without service rows:", len(unmatched_url_ids))
url_index.head()


## Processing handoff

Continue with [02_cleaning_geocoding.ipynb](02_cleaning_geocoding.ipynb)
to validate values, prepare equipment addresses, and match coordinate caches.
The observation unit and collection limitations are documented in
[the methodology](../docs/methodology.md).
